## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH10/ch10_AdaptThink.ipynb)                                             

# About this Notebook

This notebook fine-tunes **Qwen3-1.7B** to adapt how much reasoning it uses to the difficulty of a problem. Using **Group Relative Policy Optimization (GRPO)** and LoRA, the model is rewarded for answering simple mathematics problems directly while using a brief `<think>...</think>` reasoning block for more difficult problems.

A small curriculum derived from **MATH-500** separates easy and hard problems during training. The reward function combines answer correctness, appropriate use of reasoning, brevity, and proper closure of the reasoning block. A dynamic δ-controller adjusts the reasoning reward to maintain a target balance between thinking and direct answering.

The final section evaluates whether the trained model can independently decide when reasoning is useful while preserving answer accuracy and limiting unnecessary reasoning tokens.


# Dependencies

In [1]:
# Remove incompatible optional TorchAO installation.
# This notebook uses ordinary LoRA, not TorchAO quantization.
!pip uninstall -y torchao

!pip install -q -U transformers trl peft accelerate datasets

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 155.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.1 MB/s eta 0:00:00


# Imports


In [ ]:
import os, re, random, numpy as np, torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

try:
    from peft import LoraConfig
    _HAS_PEFT = True
except Exception:
    _HAS_PEFT = False

# Setup

In [ ]:


# Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Model & Think token
BASE_MODEL = "Qwen/Qwen3-1.7B"
THINK_EOS_ID = 151668  # Qwen3 </think>

# Tiny demo size
N_PER_LEVEL = 20
EASY_LEVELS = {1, 2}
HARD_LEVELS = {4, 5}

#  GRPO knobs
GRPO_GEN = 12          # 8–12 is a good balance; 12 is fine for a tiny dataset
GRPO_MAX_LEN = 256     # 256–384 for math; 256 is a good start
GRPO_TEMP = 0.7        # encourage exploration while not too random
GRPO_TOP_P = 0.9       # slightly tighter nucleus sampling

# Reward weights (will be adapted by δ-controller)
W_CORRECT = 1.5
W_GATE    = 1.0     # dynamically modulated by δ controller
W_BREVITY = 0.1
MISS_CLOSE_PENALTY = 0.1
CLOSE_BONUS = 0.05
MAX_WORDS = 40

# δ-controller (target overall no-think rate)
DELTA_TARGET = 0.60
EMA_NO_THINK = 0.60
EMA_MOMENTUM = 0.9
def update_gate_weight(batch_is_think_flags):
    global EMA_NO_THINK, W_GATE
    no_think_rate = 1.0 - (sum(batch_is_think_flags) / max(1, len(batch_is_think_flags)))
    EMA_NO_THINK = EMA_MOMENTUM*EMA_NO_THINK + (1-EMA_MOMENTUM)*no_think_rate
    err = DELTA_TARGET - EMA_NO_THINK
    # proportional tweak; clamp for stability
    W_GATE = float(np.clip(1.0 + 2.0*err, 0.4, 1.6))

# Training phases
WARMUP_EPOCHS = 1          # curriculum phase
PURE_EPOCHS   = 2          # pure AdaptThink phase

# Tokenizer & Base Policy for Debug

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)

# make sure PAD exists
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# resolve think token ids dynamically
THINK_BOS_ID = tok.convert_tokens_to_ids("<think>")
THINK_EOS_ID = tok.convert_tokens_to_ids("</think>")
assert THINK_EOS_ID not in (None, tok.unk_token_id), "Tokenizer missing </think>"

# (Optional) quick local model for spot inference before/after training
# base_policy = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True
# )


# Message Rendering Helpers

In [ ]:
SYS_HARD = (
    """This is hard. You may use a *brief* private scratchpad between <think> and </think>
    (max 8 short lines). Close </think> as soon as you spot a solution.
    After </think>, output ONLY the final answer on one line, no extra text."""
)
SYS_EASY = (
    """This is easy. Do NOT use <think>. Output ONLY the final answer in LaTeX
    as \\boxed{...} on one short line."""
)
SYS_NEUTRAL = (
    """Use <think>...</think> only when needed and CLOSE it; then output ONLY
    the final answer in LaTeX as \\boxed{...}."""
)


def build_messages(q: str, system: str):
    return [{"role": "system", "content": system},
            {"role": "user",   "content": q.strip()}]

def render_prompt_curriculum(q: str, gate: str) -> str:
    if gate == "think":
        msgs = build_messages(q, SYS_HARD)
        s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        # Seed the assistant’s turn with an opening <think> so sampling proceeds inside the block.
        return s + "<think>\n"
    else:
        msgs = build_messages(q, SYS_EASY)
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def render_prompt_pure(q: str, gate: str) -> str:
    sys = SYS_HARD if gate == "think" else SYS_EASY
    msgs = build_messages(q, sys)
    s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    return s + ("<think>\n" if gate == "think" else "")


def serialize_msgs(msgs):
    return "\n".join([f'{m["role"]}:{m["content"]}' for m in msgs])

# Small Dataset from Math-500

In [ ]:
raw = load_dataset("HuggingFaceH4/MATH-500", split="test")

def pick(levels, k):
    rows = [ex for ex in raw if int(ex["level"]) in levels]
    random.shuffle(rows)
    return rows[:k]

subset = pick(EASY_LEVELS, N_PER_LEVEL) + pick(HARD_LEVELS, N_PER_LEVEL)
random.shuffle(subset)

def normalize_ans(s: str) -> str:
    if s is None:
        return ""
    s = s.strip()

    # Prefer boxed content if present
    m = re.search(r"\\boxed\{(.+?)\}", s)
    if m:
        s = m.group(1)

    # Strip TeX wrappers and whitespace/punct
    s = (s.replace("\\left","").replace("\\right","")
           .replace("$","").replace(" ","").replace(",",""))
    # Map some common unicode / punctuation variants
    s = s.replace("−","-").replace("…","").replace("·","*")

    # Remove trailing periods/exclamations (model often prints "4.")
    s = re.sub(r"[.!]+$", "", s)

    # Normalize simple \frac{a}{b} => a/b
    s = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"\1/\2", s)

    # Normalize \sqrt{A} => sqrt(A)
    s = re.sub(r"\\sqrt\{([^{}]+)\}", r"sqrt(\1)", s)

    # Collapse multiple parentheses
    s = re.sub(r"\(+", "(", s)
    s = re.sub(r"\)+", ")", s)

    return s


def level_to_gate(level: int) -> str:
    return "no_think" if level in EASY_LEVELS else "think"

def make_rows(render_fn):
    rows = []
    for ex in subset:
        q, gold, lvl = ex["problem"], ex["answer"], int(ex["level"])
        gold_norm = normalize_ans(gold)
        gate = level_to_gate(lvl)
        prompt_str = render_fn(q, gate)     # STRING prompt
        rows.append({
            "prompt": prompt_str,
            "gold": gold_norm,
            "level": lvl,
            "gate": gate,
            "raw_q": q,
            "key": serialize_msgs(build_messages(q, SYS_NEUTRAL)),
        })
    return Dataset.from_list(rows)

ds_warmup = make_rows(render_prompt_curriculum)
ds_pure   = make_rows(render_prompt_pure)

print(f"[warmup] rows={len(ds_warmup)} easy={sum(r['gate']=='no_think' for r in ds_warmup)} hard={sum(r['gate']=='think' for r in ds_warmup)}")
print(f"[pure]   rows={len(ds_pure)}   easy={sum(r['gate']=='no_think' for r in ds_pure)} hard={sum(r['gate']=='think' for r in ds_pure)}")


# Reward with delta-controller

In [ ]:

def last_line(s: str):
    xs = [t.strip() for t in s.splitlines() if t.strip()]
    return xs[-1] if xs else s.strip()

def split_mode_pred(text: str):
    """
    Return (is_think, answer_tail, has_close).
    - If text has <think> ... </think>, 'is_think' True, 'has_close' True, answer_tail is after </think>.
    - If text contains <think> but no </think>, 'is_think' True, 'has_close' False, answer_tail is text after <think>.
    - Otherwise 'is_think' False, answer_tail is the whole text.
    """
    has_open = "<think>" in text
    has_close = "</think>" in text
    if has_open and has_close:
        pre, post = text.split("</think>", 1)
        think = pre.split("<think>", 1)[-1]
        return True, last_line(post), True
    elif has_open and not has_close:
        after_open = text.split("<think>", 1)[-1]
        return True, last_line(after_open), False
    else:
        return False, last_line(text), False


_printed_dbg = False
def adaptthink_reward(completions, prompts=None, references=None, **kwargs):
    global _printed_dbg, W_GATE

    batch = kwargs.get("batch", {})
    gates = batch.get("gate", None)
    g_refs = references or batch.get("gold", None)

    # TRL can pass either chat dicts or flat strings; normalize to strings
    comp_texts = []
    for comp in completions:
        if isinstance(comp, list):
            parts = [m["content"] for m in comp if isinstance(m, dict) and m.get("role") == "assistant"]
            comp_texts.append("\n".join(parts) if parts else comp[0]["content"])
        else:
            comp_texts.append(str(comp))

    rewards, thinks_flags = [], []
    for i, txt in enumerate(comp_texts):
        is_think, pred_raw, has_close = split_mode_pred(txt)
        pred = normalize_ans(pred_raw)
        gold = normalize_ans(g_refs[i] or "") if g_refs is not None else ""
        gate = gates[i] if gates is not None else "think"

        r = 0.0
        # correctness
        if gold:
            r += W_CORRECT * (1.0 if pred == gold else 0.0)

        # gate shaping
        if gate == "no_think":
            r += W_GATE * (1.0 if not is_think else -0.5)
            if not is_think:
                r += W_BREVITY * max(0.0, (MAX_WORDS - len(pred.split()))) / MAX_WORDS
        else:  # gate == "think"
            r += W_GATE * (1.0 if is_think else -0.5)
            if is_think and not has_close:
                r -= MISS_CLOSE_PENALTY
            if is_think and has_close:
                r += CLOSE_BONUS

        # jitter
        r += (random.random() - 0.5) * 1e-4
        r = float(np.clip(r, -1.5, 1.5))

        rewards.append(r)
        thinks_flags.append(bool(is_think))

    # delta-controller: update W_GATE based on this batch’s no-think rate
    update_gate_weight(thinks_flags)

    # first-batch debug
    if not _printed_dbg:
        mean_r = float(np.mean(rewards)) if rewards else 0.0
        pct_think = 100.0 * (sum(thinks_flags)/max(1,len(thinks_flags)))
        print(f"[reward dbg] mean={mean_r:.3f}, %think={pct_think:.1f}, W_GATE={W_GATE:.2f}, nonzero={any(abs(x)>1e-6 for x in rewards)}")
        _printed_dbg = True

    return rewards

# Setup Trainer

In [ ]:
def make_trainer(output_dir: str, dataset: Dataset, epochs: int, use_lora: bool = False):
    args = GRPOConfig(
        output_dir=output_dir,
        per_device_train_batch_size=12,
        gradient_accumulation_steps=1,
        num_train_epochs=epochs,
        max_completion_length=GRPO_MAX_LEN,
        num_generations=GRPO_GEN,
        temperature=GRPO_TEMP,
        top_p=GRPO_TOP_P,
        repetition_penalty=1.05,   # <-- new; helps close </think> and cut loops
        logging_steps=1,
    )

    peft_cfg = None
    if use_lora and _HAS_PEFT:
        peft_cfg = LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            bias="none", task_type="CAUSAL_LM"
        )

    trainer = GRPOTrainer(
        model=BASE_MODEL,
        args=args,
        reward_funcs=adaptthink_reward,
        train_dataset=dataset,
        peft_config=peft_cfg
    )
    return trainer

# Training → Pure AdaptThink

In [2]:
USE_LORA = True  # set True if you want LoRA adapters (saves VRAM)

# Warm-up
trainer = make_trainer("qwen3-1p7b-grpo-adaptthink-warmup", ds_warmup, epochs=WARMUP_EPOCHS, use_lora=USE_LORA)
print("\n=== TRAIN: WARM-UP (curriculum) ===")
trainer.train()
trainer.save_model()

# After warm-up, shift weights slightly toward correctness
W_GATE = float(np.clip(W_GATE * 0.8, 0.4, 1.6))
W_CORRECT = 1.2

# Pure AdaptThink
trainer2 = make_trainer("qwen3-1p7b-grpo-adaptthink-pure", ds_pure, epochs=PURE_EPOCHS, use_lora=USE_LORA)
print("\n=== TRAIN: PURE (both may think) ===")
trainer2.train()
trainer2.save_model()



config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



=== TRAIN: PURE (both may think) ===


Step,Training Loss
1,0.000019
2,0.021080
3,-0.000019
4,-0.000383
5,0.044974
6,-0.000038
7,0.000099
8,-0.000038
9,0.000075
10,0.000287


# Inference Helpers

In [ ]:
def decode_after_prompt(outputs, inputs):
    out_ids = outputs[0][len(inputs.input_ids[0]):].tolist()
    text = tok.decode(out_ids, skip_special_tokens=True)
    has_open = "<think>" in text
    has_close = "</think>" in text
    if has_open and has_close:
        pre, post = text.split("</think>", 1)
        thinking = pre.split("<think>", 1)[-1].strip()
        content  = post.strip()
    elif has_open:
        thinking = text.split("<think>", 1)[-1].strip()
        content  = ""
    else:
        thinking, content = "", text.strip()
    return thinking, content

def generate_with(
    policy,
    rendered_prompt: str,
    max_new_tokens=256,
    greedy=False,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.05,
):
    inputs = tok([rendered_prompt], return_tensors="pt").to(policy.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tok.pad_token_id,
        eos_token_id=[tok.eos_token_id, THINK_EOS_ID],  # stop at </think> or EOS
    )
    if greedy:
        gen_kwargs.update(dict(do_sample=False))
    else:
        gen_kwargs.update(dict(do_sample=True, temperature=temperature, top_p=top_p,
                               repetition_penalty=repetition_penalty))
    with torch.no_grad():
        outputs = policy.generate(**inputs, **gen_kwargs)
    return decode_after_prompt(outputs, inputs)

# Spot-check helpers (keep for manual sanity checks)
def answer_no_think(policy, q: str):
    rp = tok.apply_chat_template(
        build_messages(q, "This is easy. Do NOT use <think>. Answer in one short line."),
        tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    return generate_with(policy, rp, max_new_tokens=128, greedy=True)

def answer_think(policy, q: str):
    rp = tok.apply_chat_template(
        build_messages(q, "This is hard. First think in <think>...</think> and CLOSE it, then give ONLY the final answer on a new line."),
        tokenize=False, add_generation_prompt=True, enable_thinking=True
    ) + "<think>\n"  # force opening think for the spot-check only
    return generate_with(policy, rp, max_new_tokens=768, greedy=False, temperature=0.7, top_p=0.95)


# Neutral, auto-thinking (used for evaluation)
def answer_auto(policy, q: str, max_new=384, greedy=False):
    rp = tok.apply_chat_template(
        build_messages(q, SYS_NEUTRAL),
        tokenize=False, add_generation_prompt=True, enable_thinking=True
    )
    return generate_with(policy, rp, max_new_tokens=max_new, greedy=greedy, temperature=0.3, top_p=0.9)


# Load Final Policy & Spot Checks

In [ ]:
final_dir = "qwen3-1p7b-grpo-adaptthink-pure"
policy = AutoModelForCausalLM.from_pretrained(
    final_dir, device_map="auto", dtype=torch.bfloat16, trust_remote_code=True
)

def probe(prompt: str, mode: str):
    if mode == "no_think":
        t, a = answer_no_think(policy, prompt)
        print("[FORCE no-think]\nthinking:", t or "<none>", "\nanswer:", a)
    elif mode == "think":
        t, a = answer_think(policy, prompt)
        print("[FORCE think]\nthinking:", (t[:300] + "…") if t else "<none>", "\nanswer:", a)
    else:
        t, a = answer_auto(policy, prompt, greedy=False)  # or greedy=True
        print("[AUTO]\nthinking:", t or "<none>", "\nanswer:", a)

print("\n=== QUICK SPOT CHECKS ===")
probe("2 + 2 = ?", "no_think")
probe("You flip a fair coin until you see two heads in a row. What is the expected number of flips?", "think")
probe("Compute the derivative of x^3 + 2x.", "auto")


# Eval (neutral prompt; model decides thinking)

In [4]:

def answer_auto_eval(policy, q: str, max_new=384, greedy=True):
    # deterministic eval by default (greedy=True). Switch to False if you want mild sampling.
    rp = tok.apply_chat_template(
        build_messages(q, SYS_NEUTRAL),
        tokenize=False, add_generation_prompt=True, enable_thinking=True
    )
    return generate_with(policy, rp, max_new_tokens=max_new, greedy=greedy, temperature=0.3, top_p=0.9)

def eval_auto(policy, ds, n_easy=24, n_hard=24, greedy=True):
    rows = list(ds)
    easy = [r for r in rows if r["gate"] == "no_think"][:n_easy]
    hard = [r for r in rows if r["gate"] == "think"][:n_hard]

    def run(rows):
        thinks, accs, t_tok, a_tok = [], [], [], []
        for r in rows:
            t, a = answer_auto_eval(policy, r["raw_q"], greedy=greedy)
            is_think = bool(t)
            pred = normalize_ans(a.splitlines()[-1] if a.strip() else a)
            gold = normalize_ans(r["gold"])
            thinks.append(1.0 if is_think else 0.0)
            accs.append(1.0 if gold and pred == gold else 0.0)
            t_tok.append(len(tok.encode(t)) if t else 0)
            a_tok.append(len(tok.encode(a)) if a else 0)
        return float(np.mean(thinks)), float(np.mean(accs)), float(np.mean(t_tok)), float(np.mean(a_tok))

    e_think, e_acc, e_ttok, e_atok = run(easy)
    h_think, h_acc, h_ttok, h_atok = run(hard)

    print(f"[AUTO EASY]  %think={e_think*100:.1f}  acc={e_acc*100:.1f}  avg_think_tok={e_ttok:.1f}  avg_ans_tok={e_atok:.1f}")
    print(f"[AUTO HARD]  %think={h_think*100:.1f}  acc={h_acc*100:.1f}  avg_think_tok={h_ttok:.1f}  avg_ans_tok={h_atok:.1f}")

print("\n=== EVAL (AUTO ONLY) ===")
eval_auto(policy, ds_pure, n_easy=24, n_hard=24, greedy=True)



# Show last logs

print("\n=== WARMUP LOGS (last 6) ===")
for rec in trainer.state.log_history[-6:]:
    print(rec)
print("\n=== PURE LOGS (last 6) ===")
for rec in trainer2.state.log_history[-6:]:
    print(rec)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]


=== QUICK SPOT CHECKS ===
[FORCE no-think]
thinking: <none> 
answer: 4.
[FORCE think]
thinking: <none> 
answer: Okay, so I need to figure out the expected number of flips when flipping a fair coin until I get two heads in a row. Hmm, let's see. I remember that expectation problems can sometimes be approached using recursion or states. Let me try to break it down.

First, let me think about what the problem is asking. We have a fair coin, so each flip has a 50% chance of being heads (H) or tails (T). We keep flipping until we get two consecutive heads. The question is, on average, how many flips will it take?

Let me recall similar problems. For example, the expected number of flips to get one head is 2, right? Because it's a geometric distribution with p=0.5, so E = 1/p = 2. But here, we need two heads in a row. So maybe there's a similar approach but with more states.

I think the key here is to model the problem with states. Let me define some states:

- State S: Start state, no hea